# 01 - Layer 1 Multi-Stage Filtering Pipeline

This notebook covers case-study section **3.1 Layer 1: Multi-Stage Filtering Pipeline**.

It is intentionally not a broad exploratory notebook. It runs and inspects the actual project pipeline:

1. **Stage A: Hard Filters** - automatic reject rules from configurable policy thresholds.
2. **Stage B: Sector-Agnostic Benchmark Scoring** - three cross-sector exposure features plus peer outlier detection.
3. **Stage C: XGBoost Credit Model** - probability model with time-aware GridSearchCV and out-of-time validation.

The code below calls the repository scripts and then loads their generated CSVs so the outputs are visible as dataframes.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 120)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

In [ ]:
# Resolve project root whether running from repo root or notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'scripts').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

REPORTS_DIR = PROJECT_ROOT / 'reports'
MODELS_DIR = PROJECT_ROOT / 'models'
DATE_COLS = ['invoice_date', 'expected_payment_date', 'actual_resolution_date']

DATA_CANDIDATES = sorted((PROJECT_ROOT / 'utils').glob('*synthetic*.csv*'))
if not DATA_CANDIDATES:
    DATA_CANDIDATES = sorted((PROJECT_ROOT / 'utils').glob('*.csv*'))
if len(DATA_CANDIDATES) != 1:
    raise FileNotFoundError(f'Expected one dataset CSV under utils/, found: {DATA_CANDIDATES}')
DATA_PATH = DATA_CANDIDATES[0]

print('Project root:', PROJECT_ROOT)
print('Dataset:', DATA_PATH)
print('Python used by notebook:', sys.executable)

In [ ]:
def run_project_script(script_name):
    script_path = PROJECT_ROOT / 'scripts' / script_name
    result = subprocess.run(
        [sys.executable, str(script_path)],
        cwd=PROJECT_ROOT,
        check=False,
        text=True,
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()
    return result


def load_csv(name, parse_dates=None):
    return pd.read_csv(REPORTS_DIR / name, parse_dates=parse_dates)


def decision_summary(df, group_col, target_col='defaulted'):
    out = (
        df.groupby(group_col, dropna=False)
        .agg(rows=('transaction_id', 'size'))
        .reset_index()
    )
    out['row_pct'] = out['rows'] / len(df)
    if target_col in df.columns and df[target_col].notna().any():
        resolved = df[df[target_col].notna()].copy()
        resolved[target_col] = resolved[target_col].astype(int)
        outcome = (
            resolved.groupby(group_col, dropna=False)
            .agg(defaults=(target_col, 'sum'), default_rate=(target_col, 'mean'))
            .reset_index()
        )
        out = out.merge(outcome, on=group_col, how='left')
    return out

## 2. Raw Dataset Snapshot

This is only a quick baseline check so we know the denominator before Stage A removes policy rejects.

In [ ]:
raw_df = pd.read_csv(DATA_PATH, parse_dates=DATE_COLS)

raw_snapshot = pd.DataFrame({
    'metric': [
        'rows', 'columns', 'unique_exporters', 'unique_buyers',
        'paid_rows', 'defaulted_rows', 'open_rows', 'resolved_default_rate',
        'invoice_date_min', 'invoice_date_max'
    ],
    'value': [
        len(raw_df), raw_df.shape[1], raw_df['exporter_id'].nunique(), raw_df['buyer_id'].nunique(),
        int(raw_df['status'].eq('paid').sum()), int(raw_df['status'].eq('defaulted').sum()),
        int(raw_df['status'].eq('open').sum()),
        raw_df.loc[raw_df['defaulted'].notna(), 'defaulted'].mean(),
        raw_df['invoice_date'].min().date(), raw_df['invoice_date'].max().date(),
    ]
})
raw_snapshot

## 3. Stage A - Hard Filters

Stage A applies automatic reject rules before any benchmark scoring or model scoring. The rules are configurable in `config/hard_filters.json`.

In [ ]:
with open(PROJECT_ROOT / 'config' / 'hard_filters.json') as f:
    hard_filter_config = json.load(f)

pd.DataFrame(hard_filter_config['rules'])[
    ['name', 'enabled', 'column', 'operator', 'value', 'reject_reason']
]

In [ ]:
run_project_script('run_stage_a_filters.py')

stage_a_df = load_csv('stage_a_scored_transactions.csv', parse_dates=DATE_COLS)
stage_a_rule_summary = load_csv('stage_a_rule_summary.csv')

print(stage_a_df.shape)
stage_a_df.head()

In [ ]:
stage_a_decision_summary = decision_summary(stage_a_df, 'stage_a_decision')
stage_a_decision_summary

In [ ]:
stage_a_rule_summary

In [ ]:
stage_a_view_cols = [
    'transaction_id', 'invoice_date', 'status', 'defaulted',
    'stage_a_decision', 'stage_a_reject_rule_count', 'stage_a_reject_reasons',
    'docs_complete', 'exp_iscore', 'exp_auditor_tier',
    'buy_buyer_country_risk_score', 'buy_creditsafe_rating',
    'exp_historical_dispute_rate'
]

stage_a_df.loc[
    stage_a_df['stage_a_decision'].eq('REJECT_STAGE_A'),
    stage_a_view_cols
].head(15)

**Stage A interpretation checklist**

- Confirm rejected deals have clear, human-readable reasons.
- Confirm all thresholds are in JSON, not hardcoded in notebook cells.
- Confirm Stage A does not use outcome/leakage columns such as `defaulted`, `status`, `actual_resolution_date`, or `days_to_resolution`.

## 4. Stage B - Sector-Agnostic Benchmark Scoring

Stage B runs only on deals that pass Stage A. It uses exactly the three requested cross-sector features:

| Feature | Meaning |
| --- | --- |
| `ticket_to_revenue_est` | Invoice size relative to exporter revenue |
| `outstanding_to_revenue_est` | Existing Rawasi exposure relative to exporter revenue |
| `advance_tenor_exposure` | `advance_pct * payment_terms_days` |

Benchmark distributions and peer stats are fit only on Stage-A-passed training rows, then reused for validation and open rows.

In [ ]:
with open(PROJECT_ROOT / 'config' / 'benchmark_features.json') as f:
    benchmark_config = json.load(f)

pd.DataFrame(benchmark_config['features'])

In [ ]:
run_project_script('run_stage_b_benchmarks.py')

stage_b_df = load_csv('stage_b_scored_transactions.csv', parse_dates=DATE_COLS)
stage_b_feature_summary = load_csv('stage_b_feature_summary.csv')
stage_b_peer_outliers = load_csv('stage_b_peer_outliers.csv')

print(stage_b_df.shape)
stage_b_df.head()

In [ ]:
stage_b_feature_summary

In [ ]:
pd.crosstab(
    stage_b_df['stage_b_split'],
    stage_b_df['stage_b_benchmark_band'],
    margins=True
)

In [ ]:
stage_b_resolved = stage_b_df[stage_b_df['defaulted'].notna()].copy()
stage_b_resolved['defaulted'] = stage_b_resolved['defaulted'].astype(int)

stage_b_band_summary = (
    stage_b_resolved
    .groupby(['stage_b_split', 'stage_b_benchmark_band'], dropna=False)
    .agg(
        rows=('transaction_id', 'size'),
        defaults=('defaulted', 'sum'),
        default_rate=('defaulted', 'mean'),
        avg_benchmark_score=('stage_b_benchmark_score', 'mean'),
        peer_outlier_rate=('stage_b_peer_outlier_count', lambda s: (s > 0).mean()),
    )
    .reset_index()
)
stage_b_band_summary

In [ ]:
stage_b_view_cols = [
    'transaction_id', 'stage_b_split', 'status', 'defaulted',
    'goods_category',
    'ticket_to_revenue_est', 'outstanding_to_revenue_est', 'advance_tenor_exposure',
    'stage_b_ticket_to_revenue_est_risk_percentile',
    'stage_b_outstanding_to_revenue_est_risk_percentile',
    'stage_b_advance_tenor_exposure_risk_percentile',
    'stage_b_benchmark_score', 'stage_b_benchmark_band',
    'stage_b_peer_group_source', 'stage_b_peer_outlier_count', 'stage_b_peer_outlier_reasons'
]

stage_b_df[stage_b_view_cols].head(15)

In [ ]:
stage_b_peer_outliers.head(25)

In [ ]:
stage_b_df['stage_b_peer_group_source'].value_counts().to_frame('rows')

**Stage B interpretation checklist**

- `stage_b_benchmark_score` is a benchmark score, not a model probability.
- Peer outlier reasons explain which of the three features is unusual relative to the deal's `goods_category` peers.
- If a future sector has too little history, the implementation falls back to global training statistics and marks the source as `global_fallback`.

## 5. Stage C - XGBoost Credit Model

Stage C trains a binary classifier on Stage-B-enhanced, Stage-A-passed training rows.

Implementation choices:

- model: `XGBClassifier`
- tuning: `GridSearchCV`
- CV: `TimeSeriesSplit` inside the training window
- final validation: untouched out-of-time validation set
- class imbalance: `scale_pos_weight`
- primary tuning metric: average precision / AUC-PR

In [ ]:
with open(PROJECT_ROOT / 'config' / 'xgboost_model.json') as f:
    xgb_config = json.load(f)

print('Feature count:', len(xgb_config['features']))
display(pd.DataFrame({'feature': xgb_config['features']}))
print('Grid search parameter grid:')
xgb_config['grid_search']['param_grid']

In [ ]:
run_project_script('run_stage_c_model.py')

stage_c_metrics = load_csv('stage_c_metrics.csv')
stage_c_calibration = load_csv('stage_c_calibration_table.csv')
stage_c_feature_importance = load_csv('stage_c_feature_importance.csv')
stage_c_gridsearch = load_csv('stage_c_gridsearch_results.csv')
stage_c_validation_predictions = load_csv('stage_c_validation_predictions.csv', parse_dates=['invoice_date'])
stage_c_open_predictions = load_csv('stage_c_open_predictions.csv', parse_dates=['invoice_date'])

stage_c_metrics

In [ ]:
stage_c_gridsearch[[
    'rank_test_score', 'mean_test_score', 'std_test_score',
    'param_n_estimators', 'param_max_depth', 'param_learning_rate',
    'param_subsample', 'param_min_child_weight'
]].sort_values('rank_test_score').head(10)

In [ ]:
stage_c_feature_importance

In [ ]:
stage_c_calibration

In [ ]:
prediction_cols = [
    'transaction_id', 'invoice_date', 'status', 'defaulted',
    'stage_c_default_probability', 'stage_c_predicted_default',
    'stage_b_benchmark_score', 'stage_b_benchmark_band',
    'stage_b_peer_outlier_count', 'stage_b_peer_outlier_reasons'
]

stage_c_validation_predictions.sort_values(
    'stage_c_default_probability', ascending=False
)[prediction_cols].head(20)

In [ ]:
open_prediction_cols = [
    'transaction_id', 'invoice_date', 'status',
    'stage_c_default_probability',
    'stage_b_benchmark_score', 'stage_b_benchmark_band',
    'stage_b_peer_outlier_count', 'stage_b_peer_outlier_reasons'
]

stage_c_open_predictions.sort_values(
    'stage_c_default_probability', ascending=False
)[open_prediction_cols].head(20)

## 6. Stage C Plots

In [ ]:
for image_name in [
    'stage_c_precision_recall.png',
    'stage_c_calibration.png',
    'stage_c_confusion_matrix.png',
    'stage_c_feature_importance.png',
]:
    display(Markdown(f'### {image_name}'))
    display(Image(filename=str(REPORTS_DIR / image_name)))

## 7. Layer 1 Summary

This notebook now maps directly to section 3.1:

| Requirement | Where shown |
| --- | --- |
| Stage A configurable hard filters | `config/hard_filters.json`, Stage A rule table, Stage A decision output |
| Stage B cross-sector features | Stage B feature config and feature summary |
| Within-sector outlier detection | Stage B peer outlier table and reason strings |
| New/sparse sector handling | `stage_b_peer_group_source`, fallback logic documented in Stage B notes |
| Stage C probability classifier | XGBoost model outputs and validation predictions |
| Class imbalance | `scale_pos_weight` in Stage C report |
| Out-of-time validation | train/validation split outputs and Stage C validation metrics |
| AUC-PR / KS | `stage_c_metrics` |
| Calibration plot/table | Stage C calibration table and plot |
| Explainability | XGBoost feature importance plus Stage B human-readable benchmark/outlier drivers |
| Confusion matrix and threshold | Stage C confusion matrix plot and metrics table |

Key caveat: the current Stage C model uses only the three requested Stage B exposure features and their derived benchmark/outlier features. It is useful for risk ranking, but the calibration table shows it should not yet be treated as a pricing-grade default probability model.